<a href="https://colab.research.google.com/github/quocthescaredycat/hub-banking-support-ticket-routing/blob/main/training_transformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BANKING77 — explicit PyTorch fine-tuning

Load BANKING77 with Hugging Face, then train `DistilBERT` or `RoBERTa` from `model_lib.py`. Each is an `nn.Module` with a pretrained encoder, first-token selection, and an explicit classification head. The batch loop below shows forward propagation, cross-entropy, backpropagation, and optimizer updates.

Install once: `%pip install datasets transformers torch scikit-learn pandas`.

Use the same 90/10 development split as `training.ipynb`. Official test scoring comes after model selection; the normalized duplicate overlap is documented in `preprocess.ipynb`.

In [1]:
!pip install datasets transformers torch scikit-learn pandas

In [2]:
import json
import math
import unicodedata
from pathlib import Path

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader
from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score
from transformers import AutoTokenizer, DataCollatorWithPadding, get_linear_schedule_with_warmup, set_seed

from model_lib import DistilBERT, RoBERTa

## 1. Hyperparameters

In [3]:
checkpoint = "distilbert/distilbert-base-uncased"  # or "FacebookAI/roberta-base"
examples_per_intent = None  # 5, 30, or None for all fitting examples
seed = 42
split_seed = 42  # keep fixed across experiments

batch_size = 32
learning_rate = 4e-5
weight_decay = 0.01
epochs = {5: 50, 30: 18, None: 5}[examples_per_intent]  # or set an integer
max_length = 128
warmup_ratio = 0.1
max_grad_norm = 1.0
dropout = 0.2 if checkpoint == "distilbert/distilbert-base-uncased" else 0.1
bf16 = True  # enabled only on supported CUDA GPUs

run_name = f"{checkpoint.split('/')[-1]}-{examples_per_intent or 'all'}-seed{seed}-pytorch"
output = Path("artifacts/transformer") / run_name

set_seed(seed)

## 2. Load and split

The dataset supplies the integer labels and their names. Keep the split seed fixed when changing training or few-shot sampling seeds.

Only the official training set is split: 9,002 training and 1,001 validation requests. The 3,080 official test requests remain separate as `test_ds`. Hugging Face calls the new validation partition `split["test"]`; this is different from `ds["test"]`.

In [4]:
ds = load_dataset("PolyAI/banking77")
label_names = ds["train"].features["label"].names

split = ds["train"].train_test_split(test_size=0.1, stratify_by_column="label", seed=split_seed)
train_ds = split["train"]
valid_ds = split["test"]
test_ds = ds["test"]
print(f"Train: {len(train_ds):,} | Validation: {len(valid_ds):,} | Official test: {len(test_ds):,} | Intents: {len(label_names)}")

README.md:   0%|          | 0.00/13.0k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  294kB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 92.4kB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/10003 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3080 [00:00<?, ? examples/s]

Train: 9,002 | Validation: 1,001 | Official test: 3,080 | Intents: 77


## 3. Choose the model and label budget

Use `examples_per_intent = None` for the full fitting partition, `5` for 385 labeled requests, or `30` for 2,310. The smaller subsets are nested at a fixed seed and sampled only from training; validation stays fixed. `None` means all **fitting** rows, not all 10,003 official training rows.

Few-shot runs measure how much labeled data a pretrained model needs. More epochs give small datasets more optimization passes; they do not guarantee better accuracy. The 50/18/5 epoch schedule and optimizer settings below follow [Qi et al., §4](https://arxiv.org/html/2012.03929v2). Our held-out validation and checkpoint selection make this a paper-informed experiment, not an exact reproduction.

In [5]:
fit_ds = train_ds
if examples_per_intent is not None:
    shuffled = train_ds.shuffle(seed=seed)
    labels = np.asarray(shuffled["label"])
    indices = np.concatenate([
        np.flatnonzero(labels == label)[:examples_per_intent]
        for label in range(len(label_names))
    ])
    fit_ds = shuffled.select(indices.tolist()).shuffle(seed=seed)

print(f"{checkpoint} | {len(fit_ds):,} training examples | {epochs} epochs")

distilbert/distilbert-base-uncased | 9,002 training examples | 5 epochs


## 4. Tokenize and build data loaders

Light cleaning preserves case, punctuation, numbers, and negation. The checkpoint's own tokenizer handles subwords and special tokens. Truncate at 128 tokens; pad to the longest request in each batch. `DataLoader` feeds dictionaries of tensors into the model.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def clean_text(text):
    return " ".join(unicodedata.normalize("NFC", text).split())

def tokenize(batch):
    return tokenizer([clean_text(text) for text in batch["text"]],
                     truncation=True, max_length=max_length, return_token_type_ids=False)

train_tokens = fit_ds.map(tokenize, batched=True, remove_columns=["text"]).rename_column("label", "labels")
valid_tokens = valid_ds.map(tokenize, batched=True, remove_columns=["text"]).rename_column("label", "labels")
collator = DataCollatorWithPadding(tokenizer=tokenizer)
train_loader = DataLoader(train_tokens, batch_size=batch_size, shuffle=True, collate_fn=collator)
valid_loader = DataLoader(valid_tokens, batch_size=batch_size, shuffle=False, collate_fn=collator)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/9002 [00:00<?, ? examples/s]

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

## 5. Create the classifier

Inspect the two classes in `model_lib.py`. RoBERTa selects the first `<s>` token and uses a dense/tanh head; DistilBERT selects `[CLS]` and uses a dense/ReLU head. Both return `[batch_size, 77]` logits. All encoder and head parameters are trainable.

```text
input IDs + attention mask → pretrained encoder → first token → classifier → logits
```

`CrossEntropyLoss` handles log-softmax internally. Apply softmax when displaying prediction probabilities.

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
model_class = {
    "distilbert/distilbert-base-uncased": DistilBERT,
    "FacebookAI/roberta-base": RoBERTa,
}[checkpoint]
model = model_class(num_labels=len(label_names), checkpoint=checkpoint, dropout=dropout).to(device)
use_bf16 = bf16 and device.type == "cuda" and torch.cuda.is_bf16_supported()

print("Device:", device, "| BF16:", use_bf16)
print(model.classifier)
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Device: cuda | BF16: True
Sequential(
  (0): Linear(in_features=768, out_features=768, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=768, out_features=77, bias=True)
)
Trainable parameters: 67,012,685


## 6. Optimizer and validation

AdamW, the data loaders, and the linear warmup scheduler use the hyperparameters from the first configuration cell. Epochs remain 50/18/5 for 5-shot/30-shot/all fitting examples, following the paper cited above. BF16 autocasting is enabled on supported CUDA GPUs, including the RTX 4090; other devices use FP32.

Validation runs without gradients and reports sample-averaged loss, accuracy, and macro-F1 over all 77 labels.

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
total_steps = epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=math.ceil(warmup_ratio * total_steps), num_training_steps=total_steps,
)

@torch.inference_mode()
def evaluate(model, loader):
    model.eval()
    total_loss, targets, all_logits = 0.0, [], []
    for batch in loader:
        labels = batch.pop("labels").to(device)
        inputs = {key: value.to(device) for key, value in batch.items()}
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(**inputs)
            loss = criterion(logits, labels)
        total_loss += loss.item() * len(labels)
        targets.extend(labels.tolist())
        all_logits.append(logits.float().cpu())
    targets = np.asarray(targets)
    logits = torch.cat(all_logits).numpy()
    predictions = logits.argmax(axis=-1)
    metrics = {
        "loss": total_loss / len(targets),
        "accuracy": accuracy_score(targets, predictions),
        "macro_f1": f1_score(targets, predictions, labels=range(len(label_names)),
                             average="macro", zero_division=0),
    }
    return metrics, targets, logits

## 7. Train and save the best checkpoint

Each batch computes logits and loss, clears gradients, backpropagates, clips gradient norms, and updates the optimizer and scheduler. Validation macro-F1 selects the checkpoint; the best weights are restored afterward.

The checkpoint stores model weights, backbone name, label names, sampling budget, seed, and epoch. It is a PyTorch checkpoint for these custom classes. Runs write to a separate `-pytorch` directory; rerunning the same configuration replaces its outputs.

In [9]:
output.mkdir(parents=True, exist_ok=True)
tokenizer.save_pretrained(output / "tokenizer")
best_f1, history = -1.0, []

for epoch in range(epochs):
    model.train()
    total_loss = 0.0
    for batch in train_loader:
        labels = batch.pop("labels").to(device)
        inputs = {key: value.to(device) for key, value in batch.items()}
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.bfloat16, enabled=use_bf16):
            logits = model(**inputs)
            loss = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item() * len(labels)

    metrics, _, _ = evaluate(model, valid_loader)
    history.append({"epoch": epoch + 1, "train_loss": total_loss / len(train_loader.dataset), **metrics})
    print(f"Epoch {epoch + 1}/{epochs} | train loss {history[-1]['train_loss']:.4f} | "
          f"val loss {metrics['loss']:.4f} | accuracy {metrics['accuracy']:.4f} | macro-F1 {metrics['macro_f1']:.4f}")
    if metrics["macro_f1"] > best_f1:
        best_f1 = metrics["macro_f1"]
        torch.save({"model_state_dict": model.state_dict(), "checkpoint": checkpoint,
                    "label_names": label_names, "examples_per_intent": examples_per_intent,
                    "seed": seed, "epoch": epoch + 1,
                    "hyperparameters": {"batch_size": batch_size, "learning_rate": learning_rate,
                        "weight_decay": weight_decay, "epochs": epochs, "max_length": max_length,
                        "warmup_ratio": warmup_ratio, "max_grad_norm": max_grad_norm,
                        "dropout": dropout, "bf16": bf16, "split_seed": split_seed}}, output / "best.pt")

(output / "history.json").write_text(json.dumps(history, indent=2), encoding="utf-8")
model.load_state_dict(torch.load(output / "best.pt", map_location="cpu", weights_only=True)["model_state_dict"])
print("Best validation metrics:", evaluate(model, valid_loader)[0])

Epoch 1/5 | train loss 3.3638 | val loss 1.6549 | accuracy 0.7383 | macro-F1 0.6761
Epoch 2/5 | train loss 1.0512 | val loss 0.5967 | accuracy 0.8781 | macro-F1 0.8636
Epoch 3/5 | train loss 0.4206 | val loss 0.4193 | accuracy 0.8971 | macro-F1 0.8901
Epoch 4/5 | train loss 0.2309 | val loss 0.3582 | accuracy 0.9041 | macro-F1 0.9012
Epoch 5/5 | train loss 0.1514 | val loss 0.3465 | accuracy 0.9091 | macro-F1 0.9067
Best validation metrics: {'loss': 0.3465109109603144, 'accuracy': 0.9090909090909091, 'macro_f1': 0.9067113608247912}


## 8. Try the trained model

Use the same cleaning and tokenizer, then convert logits into probabilities. These probabilities have not been calibrated for automatic routing.

For a fresh session, instantiate the class using the checkpoint's backbone name, label count, and saved head dropout, load `model_state_dict`, and load the tokenizer from the saved `tokenizer` directory.

In [10]:
requests = ["My card still hasn't arrived.", "I don't recognise this card payment."]
inputs = tokenizer([clean_text(text) for text in requests], padding=True, truncation=True,
                   max_length=max_length, return_token_type_ids=False, return_tensors="pt").to(device)
model.eval()
with torch.inference_mode():
    probabilities = model(**inputs).softmax(dim=-1)
confidence, predicted = probabilities.max(dim=-1)
list(zip(requests, [label_names[i] for i in predicted.tolist()], confidence.tolist()))

[("My card still hasn't arrived.", 'card_arrival', 0.9755728840827942),
 ("I don't recognise this card payment.",
  'card_payment_not_recognised',
  0.9838159680366516)]

## 9. Final evaluation on the official test set

After freezing settings, load the current run's `best.pt` and saved tokenizer explicitly. `output` is defined in the configuration cell from the backbone, label budget, and seed; select that run before loading. To evaluate an existing checkpoint after a kernel restart, run the setup cells and skip the training cell. In an already-running kernel, rerun section 6 first to refresh `evaluate`, which now returns metrics, labels, and logits.

Restore the backbone name, label names, head dropout, maximum length, batch size, and precision preference from the checkpoint. Score the official test set once with the shared `evaluate` function, which runs without gradients. Save test metrics and the aligned labels, predictions, and logits in the same run directory.

This scores the model trained on the 90% fitting partition. Published full-data experiments train on all official training examples. Keep official test results out of further hyperparameter and checkpoint selection.

In [14]:
saved_path = output / "best.pt"
saved = torch.load(saved_path, map_location="cpu", weights_only=True)
test_config = saved["hyperparameters"]
tokenizer = AutoTokenizer.from_pretrained(output / "tokenizer")
label_names = saved["label_names"]
max_length = test_config["max_length"]
use_bf16 = test_config["bf16"] and device.type == "cuda" and torch.cuda.is_bf16_supported()

test_model_class = {
    "distilbert/distilbert-base-uncased": DistilBERT,
    "FacebookAI/roberta-base": RoBERTa,
}[saved["checkpoint"]]
test_model = test_model_class(
    num_labels=len(label_names), checkpoint=saved["checkpoint"], dropout=test_config["dropout"],
).to(device)
test_model.load_state_dict(saved["model_state_dict"])

test_tokens = test_ds.map(tokenize, batched=True, remove_columns=["text"]).rename_column("label", "labels")
test_loader = DataLoader(test_tokens, batch_size=test_config["batch_size"], shuffle=False,
                         collate_fn=DataCollatorWithPadding(tokenizer=tokenizer))

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

In [15]:
test_metrics, test_labels, test_logits = evaluate(test_model, test_loader)
test_metrics = {f"test_{name}": value for name, value in test_metrics.items()}
(output / "test_results.json").write_text(json.dumps(test_metrics, indent=2), encoding="utf-8")
np.savez_compressed(
    output / "test_predictions.npz",
    labels=test_labels,
    predictions=test_logits.argmax(axis=-1),
    logits=test_logits,
)
test_metrics

{'test_loss': 0.3183512105570211,
 'test_accuracy': 0.9204545454545454,
 'test_macro_f1': 0.9200519983892261}

References: [Hugging Face native PyTorch fine-tuning](https://huggingface.co/docs/transformers/v4.40.2/training), [PyTorch cross-entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html), [Qi et al.](https://arxiv.org/html/2012.03929v2).